### 1. Import Libraries & Mount Drive
Import required machine learning and data processing libraries.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Kaggle-friendly path setup
if os.path.exists('/kaggle/input'):
    data_dir = '/kaggle/input/nilm-refit-house2-processed'  # Update dataset name if different
else:
    data_dir = '../../data/processed_data'  # Local fallback


### 2. Load Processed Data
Load the concatenated chunks of the processed house dataset.

In [ ]:
# Đọc file CSV duy nhất đã xử lý sẵn (parse_dates ngay khi đọc để tránh phải convert lại)
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))
assert len(csv_files) > 0, f"Không tìm thấy file CSV nào trong {data_dir}"
csv_path = csv_files[0]
print(f"Loading: {csv_path}")

appliance_cols = [f'Appliance{i}' for i in range(1, 10)]
float_cols = ['Aggregate'] + appliance_cols

# Khai báo dtype trước -> giảm RAM (~50%) và tăng tốc RandomForest sau này (float32 thay float64)
dtype_map = {col: 'float32' for col in float_cols}
dtype_map.update({'Hour': 'int8', 'DayOfWeek': 'int8'})

df = pd.read_csv(
    csv_path,
    parse_dates=['Time'],
    dtype=dtype_map,
)

# Phòng trường hợp file CSV chưa có sẵn cột Hour/DayOfWeek
if 'Hour' not in df.columns:
    df['Hour'] = df['Time'].dt.hour.astype('int8')
if 'DayOfWeek' not in df.columns:
    df['DayOfWeek'] = df['Time'].dt.dayofweek.astype('int8')

print(f"df shape: {df.shape}, memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()


### 3. Feature Engineering & Train-Test Split
Adding time-based features (`Is_Weekend`, `Time_of_Day`) to help the Random Forest capture human behavioral patterns.

In [ ]:
# 1. Cuối tuần (Thứ 7, CN) -> Trả về 1 (Có) hoặc 0 (Không)
df['Is_Weekend'] = df['Time'].dt.dayofweek.isin([5, 6]).astype('int8')

# 2. Chia 4 buổi trong ngày — vector hóa bằng np.select thay cho .apply() (nhanh hơn nhiều trên dataset lớn)
conditions = [
    (df['Hour'] >= 5) & (df['Hour'] < 8),    # Sáng (Morning)
    (df['Hour'] >= 8) & (df['Hour'] < 17),   # Ngày (Daytime)
    (df['Hour'] >= 17) & (df['Hour'] < 22),  # Tối (Evening)
]
choices = [1, 2, 3]
df['Time_of_Day'] = np.select(conditions, choices, default=0).astype('int8')  # default 0 = Đêm (Night)

# 2. Định nghĩa tập X (Time features + Aggregate hiện tại)
X = df[['Hour', 'DayOfWeek', 'Is_Weekend', 'Time_of_Day', 'Aggregate']]
time_index = df['Time'] # Giữ lại để lát vẽ biểu đồ

# 3. Định nghĩa tập y (10 cột)
list_9_appliances = ['Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']

# Tính cột thứ 10: Unknown_Appliance
# Lưu ý: Đôi khi do nhiễu đo lường, tổng 9 thiết bị có thể lớn hơn Aggregate sinh ra số âm. 
# Dùng .clip(lower=0) để đưa các số âm về 0 cho chuẩn thực tế.
df['Unknown_Appliance'] = (df['Aggregate'] - df[list_9_appliances].sum(axis=1)).clip(lower=0)

y = df[list_9_appliances + ['Unknown_Appliance']]

# 4. Chia Train/Test (Vì là chuỗi thời gian, ta KHÔNG xáo trộn dữ liệu - shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
time_train, time_test = train_test_split(time_index, test_size=0.2, shuffle=False)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


### 4. Train Random Forest Regressor & Predict
Training an ensemble of decision trees to map aggregate power and time features to individual appliance powers.

In [18]:
# ⚠️ Lưu ý tốc độ trên Kaggle: notebook này nên chạy với Accelerator = "None" (CPU only).
# RandomForestRegressor của sklearn KHÔNG dùng GPU. Máy GPU trên Kaggle chỉ có 2 CPU core,
# trong khi máy CPU-only có 4 CPU core -> bật GPU ở đây sẽ làm n_jobs=-1 chậm hơn, không nhanh hơn.

# Sử dụng Random Forest, có thể tuning n_estimators hoặc max_depth
model = RandomForestRegressor(n_estimators=100, max_depth=10,min_samples_leaf=5,max_features=None, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Dự đoán
y_pred = model.predict(X_test)

# Bất kỳ dự đoán nào < 5W (hoặc một ngưỡng nhỏ), ta ép thẳng về 0W
THRESHOLD = 5.0
y_pred[y_pred < THRESHOLD] = 0.0






### 5. Evaluate Metrics (Energy-based & MAE)
Calculate energy-based precision, recall, F1, NEP, and MAE for all appliances to compare with other baselines.

In [19]:
def calculate_nilm_metrics(y_true, y_pred, appliance_names):
    """
    Calculate Energy-based metrics and MAE Precision, Recall, F1-score và NEP
    dựa trên công thức học thuật.
    """
    # Đảm bảo đầu vào là numpy array để tránh lỗi lệch index của Pandas
    y_t = np.array(y_true)
    y_p = np.array(y_pred)
    
    # Cộng thêm 1e-9 (số cực nhỏ) vào mẫu số để tránh lỗi Division by Zero (chia cho 0)
    # vì có những thiết bị tắt hoàn toàn trong tập test (tổng năng lượng = 0)
    eps = 1e-9 
    
    # Tính tử số chung của Precision và Recall: sum(min(y_pred, y_true))
    min_power = np.minimum(y_p, y_t)
    sum_min_power = np.sum(min_power, axis=0) # Cộng dồn theo trục thời gian (T)
    
    # Tính mẫu số
    sum_pred = np.sum(y_p, axis=0) # Tổng điện dự đoán
    sum_true = np.sum(y_t, axis=0) # Tổng điện thực tế
    
    # 1. Energy-based Precision (PE)
    P_E = sum_min_power / (sum_pred + eps)
    
    # 2. Energy-based Recall (RE)
    R_E = sum_min_power / (sum_true + eps)
    
    # 3. Energy-based F1-score (FE)
    F1_E = 2 * (P_E * R_E) / (P_E + R_E + eps)
    
    # 4. Normalized Error in Assigned Power (NEP)
    sum_abs_error = np.sum(np.abs(y_t - y_p), axis=0)
    NEP = sum_abs_error / (sum_true + eps)
    
    # 5. Mean Absolute Error (MAE)
    MAE = sum_abs_error / y_t.shape[0]
    
    # --- Package results into a DataFrame (DataFrame) cho dễ nhìn ---
    results_df = pd.DataFrame({
        'Appliance': appliance_names,
        'Precision (PE)': np.round(P_E, 4),
        'Recall (RE)': np.round(R_E, 4),
        'F1-Score (FE)': np.round(F1_E, 4),
        'NEP': np.round(NEP, 4),
        'MAE (W)': np.round(MAE, 4)
    })
    
    # Add Macro Average row (Macro Average) của cả nhà
    mean_row = pd.DataFrame({
        'Appliance': ['--- AVERAGE ---'],
        'Precision (PE)': [np.round(np.mean(P_E), 4)],
        'Recall (RE)': [np.round(np.mean(R_E), 4)],
        'F1-Score (FE)': [np.round(np.mean(F1_E), 4)],
        'NEP': [np.round(np.mean(NEP), 4)],
        'MAE (W)': [np.round(np.mean(MAE), 4)]
    })
    
    results_df = pd.concat([results_df, mean_row], ignore_index=True)
    return results_df

# Danh sách tên 10 thiết bị (lấy từ các bước trước)
appliance_names = list_9_appliances + ['Unknown_Appliance']

# Gọi hàm tính điểm
metrics_table = calculate_nilm_metrics(y_test, y_pred, appliance_names)

# Display metrics table
print("RANDOM FOREST MODEL RATING TABLE:")
display(metrics_table)


BẢNG ĐÁNH GIÁ MÔ HÌNH DỰA TRÊN NĂNG LƯỢNG (ENERGY-BASED METRICS):


,Appliance,Precision (PE),Recall (RE),F1-Score (FE),NEP
0,Appliance1,0.6361,0.6902,0.6621,0.7046
1,Appliance2,0.2257,0.2686,0.2453,1.6530
2,Appliance3,0.5506,0.5569,0.5537,0.8976
3,Appliance4,0.2525,0.3778,0.3027,1.7408
4,Appliance5,0.3958,0.2773,0.3261,1.1460
5,Appliance6,0.5233,0.5141,0.5186,0.9543
6,Appliance7,0.0850,0.0188,0.0308,1.1838
7,Appliance8,0.7296,0.6817,0.7049,0.5709
8,Appliance9,0.0151,0.0486,0.0230,4.1214
9,Unknown_Appliance,0.8873,0.8664,0.8767,0.2436


### 6. Visualization
Plotting the predicted vs actual power consumption for the test sequence.

In [ ]:
def plot_10_appliances(y_test, y_pred, time_test):
    appliance_names = list_9_appliances + ['Unknown_Appliance']
    
    # Tạo 10 subplots (5 hàng, 2 cột)
    fig, axes = plt.subplots(5, 2, figsize=(16, 20))
    axes = axes.flatten()
    
    # Lấy khoảng 1000 điểm dữ liệu đầu tiên của tập test để biểu đồ không bị rối mớ bong bong
    plot_length = 1000 
    
    for i in range(10):
        # Vẽ giá trị thực tế
        axes[i].plot(time_test.iloc[:plot_length], y_test.iloc[:plot_length, i], 
                     label='Actual (Actual)', color='blue', alpha=0.6)
        # Vẽ giá trị dự đoán
        axes[i].plot(time_test.iloc[:plot_length], y_pred[:plot_length, i], 
                     label='Predicted (Predicted)', color='red', alpha=0.6, linestyle='--')
        
        axes[i].set_title(f'Appliance: {appliance_names[i]}')
        axes[i].set_ylabel('Power (W)')
        axes[i].legend(loc='upper right')
        
        # Format lại trục X cho dễ nhìn
        plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.show()

plot_10_appliances(y_test, y_pred, time_test)